# MammosighTR — EfficientNet-B0 BI-RADS Sınıflandırması (GeM + Kalite Öznitelik Füzyonu)

Bu çalışma, mamografi görüntülerinin sınıflandırılması amacıyla Genelleştirilmiş Ortalama Havuzlama (GeM) ve 12-boyutlu görüntü kalitesi/pozisyonlama öznitelik yan kolu ile güçlendirilmiş **EfficientNet-B0** mimarisinin eğitim ve değerlendirme adımlarını içermektedir.

### Metodoloji ve Mimari Detaylar:
1. **Model Mimarisi:** EfficientNet-B0 omurgası, GeM + MaxPool hibrit havuzlama, 12-B kalite öznitelik MLP yan kolu ve Multi-Sample Dropout (5x).
2. **Eğitim Stratejisi:** Üstel Hareketli Ortalama (ModelEMA), Focal Cross-Entropy kayıp fonksiyonu, Cosine Annealing öğrenme oranı çizelgesi.
3. **Değerlendirme:** Çok ölçekli TTA, Pareto eşik kalibrasyonu ve hasta düzeyinde çoklu projeksiyon bağlamsal füzyonu.


In [ ]:
# ==============================================================================
# Bölüm 1: Ortam Konfigürasyonu, Bağımlılıklar ve Başlatma
# ==============================================================================
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'albumentations', 'scikit-learn', 'scipy'], check=False)

import os, re, glob, time, json, random, shutil, zipfile
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from scipy.optimize import minimize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import timm
from timm.utils import ModelEmaV2
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    roc_auc_score, f1_score, accuracy_score
)
from sklearn.preprocessing import label_binarize

SESSION_START_TIME = time.time()
MAX_SESSION_SECONDS = int(11.2 * 3600)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count()
print(f'PyTorch: {torch.__version__} | Cihaz: {DEVICE} ({NUM_GPUS} GPU)')


In [ ]:
# ── 2. Veri Seti Çıkarma & Önceki Checkpoint / Önbellek Kurtarma ──
DATASET_DIR   = '/kaggle/temp/yolo_dataset' if os.path.exists('/kaggle') else './yolo_dataset'
SAVE_DIR      = '/kaggle/working/runs/efficientnet_b0' if os.path.exists('/kaggle') else './runs/efficientnet_b0'
LAST_CKPT     = os.path.join(SAVE_DIR, 'last_checkpoint.pth')
BEST_MODEL_PT = os.path.join(SAVE_DIR, 'best_model.pth')
RESULTS_CSV   = os.path.join(SAVE_DIR, 'results.csv')
STATE_JSON    = os.path.join(SAVE_DIR, 'stage_state.json')
Q_CACHE_NPZ   = os.path.join(SAVE_DIR, 'quality_features_cache.npz')
TTA_CACHE_NPZ = os.path.join(SAVE_DIR, 'efficientnet_tta_cache.npz')
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

# Önceki oturumdan (/kaggle/input) kaydedilen tüm ilerlemeleri geri yükle
if os.path.exists('/kaggle/input'):
    for fname, dst_path in [
        ('last_checkpoint.pth', LAST_CKPT),
        ('best_model.pth', BEST_MODEL_PT),
        ('results.csv', RESULTS_CSV),
        ('stage_state.json', STATE_JSON),
        ('quality_features_cache.npz', Q_CACHE_NPZ),
        ('efficientnet_tta_cache.npz', TTA_CACHE_NPZ),
    ]:
        if not os.path.exists(dst_path):
            matches = [m for m in glob.glob(f'/kaggle/input/**/{fname}', recursive=True) if 'efficientnet' in m.lower() or 'quality_features' in m.lower()]
            if not matches and fname in ('last_checkpoint.pth', 'best_model.pth', 'quality_features_cache.npz', 'efficientnet_tta_cache.npz'):
                matches = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
            if matches:
                shutil.copy2(matches[0], dst_path)
                print(f'Önceki oturumdan geri yüklendi: {matches[0]} -> {dst_path}')

if not os.path.exists(os.path.join(DATASET_DIR, 'train')):
    zip_Search = (
        glob.glob('/kaggle/input/**/yolo_dataset.zip', recursive=True) +
        glob.glob('/kaggle/working/**/yolo_dataset.zip', recursive=True) +
        glob.glob('./**/yolo_dataset.zip', recursive=True)
    )
    if zip_Search:
        zip_path = zip_Search[0]
        print(f'Veri arşivi bulundu: {zip_path}, {DATASET_DIR} dizinine çıkartılıyor...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(DATASET_DIR)
        if not os.path.exists(os.path.join(DATASET_DIR, 'train')):
            sub_trains = glob.glob(os.path.join(DATASET_DIR, '*/train'))
            if sub_trains:
                DATASET_DIR = os.path.dirname(sub_trains[0])
        print(f'Arşiv çıkartıldı. Aktif dizin: {DATASET_DIR}')
    else:
        dir_Search = [d for d in glob.glob('/kaggle/input/**/yolo_dataset', recursive=True) if os.path.isdir(d)]
        if dir_Search:
            DATASET_DIR = dir_Search[0]
            print(f'Hazır klasör bulundu: {DATASET_DIR}')
        else:
            raise FileNotFoundError('yolo_dataset.zip bulunamadı!')

for split in ['train', 'val', 'test']:
    split_path = os.path.join(DATASET_DIR, split)
    for cls_Old, cls_New in [('BIRADS0', 'BIRADS_0'), ('BIRADS1_2', 'BIRADS_1_2'), ('BIRADS4_5', 'BIRADS_4_5')]:
        p_old = os.path.join(split_path, cls_Old)
        p_new = os.path.join(split_path, cls_New)
        if os.path.exists(p_old) and not os.path.exists(p_new):
            os.rename(p_old, p_new)

CLASS_NAMES   = ['BIRADS_0', 'BIRADS_1_2', 'BIRADS_4_5']
DISPLAY_NAMES = ['BI-RADS 0 (Ek Tetkik)', 'BI-RADS 1-2 (Normal/Benign)', 'BI-RADS 4-5 (Şüpheli)']
for split in ['train', 'val', 'test']:
    counts = {cls: len(glob.glob(f'{DATASET_DIR}/{split}/{cls}/*.jpg')) for cls in CLASS_NAMES}
    print(f'{split:5s}: {counts} (Toplam: {sum(counts.values())})')


In [ ]:
# ==============================================================================
# Bölüm 3: 12-Boyutlu Kalite/Pozisyonlama Öznitelik Çıkarımı ve Veri Yükleyiciler
# ==============================================================================
IMG_SIZE    = 512
BATCH_SIZE  = 32 * max(1, NUM_GPUS)
NUM_WORKERS = 4

def extract_quality_positioning_features(img_gray):
    """12-boyutlu görüntü kalitesi ve anatomik pozisyonlama öznitelik vektörü çıkarımı.

    Öznitelikler: Yoğunluk istatistikleri (ortalama, varyans, entropi), yerel kontrast,
    Sobel kenar gradyanları, Laplacian keskinlik skoru, kütle merkezi ve doku doluluk oranı.
    """
    h, w = img_gray.shape[:2]
    fg = img_gray > 5
    fg_ratio = float(fg.mean())
    if fg_ratio < 0.01:
        return np.zeros(12, dtype=np.float32)

    fg_pixels = img_gray[fg].astype(np.float32) / 255.0
    mean_int = float(fg_pixels.mean())
    std_int  = float(fg_pixels.std())
    p90_int  = float(np.percentile(fg_pixels, 90))
    dense_ratio = float((fg_pixels > 0.65).mean())

    hist = cv2.calcHist([img_gray], [0], fg.astype(np.uint8), [32], [1, 256]).flatten()
    hist = hist / (hist.sum() + 1e-7)
    entropy = float(-np.sum(hist * np.log2(hist + 1e-7)) / 5.0)

    lap = cv2.Laplacian(img_gray, cv2.CV_32F)
    lap_var = float(np.log1p(lap[fg].var()) / 10.0)
    gx = cv2.Sobel(img_gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_gray, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = np.hypot(gx, gy)
    edge_mean = float(np.log1p(grad_mag[fg].mean()) / 6.0)

    left_edge_mass  = float(fg[:, :max(1, int(w * 0.08))].mean())
    right_edge_mass = float(fg[:, min(w - 1, int(w * 0.92)):].mean())
    chest_wall_mass = max(left_edge_mass, right_edge_mass)
    upper_quad_mass = float(fg[:int(h * 0.35), :].mean())
    lower_quad_mass = float(fg[int(h * 0.65):, :].mean())

    ys, xs = np.where(fg)
    bbox_h = (ys.max() - ys.min() + 1) / float(h)
    bbox_w = (xs.max() - xs.min() + 1) / float(w)
    bbox_fill = float(fg_ratio / max(bbox_h * bbox_w, 1e-4))

    return np.array([
        fg_ratio, mean_int, std_int, p90_int, dense_ratio, entropy,
        lap_var, edge_mean, chest_wall_mass, upper_quad_mass, lower_quad_mass, bbox_fill
    ], dtype=np.float32)

def scan_split_files(root_dir):
    samples, labels, cases, views = [], [], [], []
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        cls_dir = os.path.join(root_dir, cls_name)
        if not os.path.exists(cls_dir): continue
        for fpath in sorted(glob.glob(os.path.join(cls_dir, '*.jpg'))):
            bname = os.path.basename(fpath)
            m_case = re.search(r'case(\d+)', bname)
            m_view = re.search(r'_(RCC|LCC|RMLO|LMLO)_', bname)
            samples.append(fpath)
            labels.append(cls_idx)
            cases.append(int(m_case.group(1)) if m_case else -1)
            views.append(m_view.group(1) if m_view else 'UNK')
    return samples, np.array(labels, dtype=np.int64), np.array(cases, dtype=np.int64), np.array(views)

train_samples, train_labels, train_cases, train_views = scan_split_files(f'{DATASET_DIR}/train')
val_samples, val_labels, val_cases, val_views         = scan_split_files(f'{DATASET_DIR}/val')
test_samples, test_labels, test_cases, test_views     = scan_split_files(f'{DATASET_DIR}/test')

# 12-boyutlu görüntü kalitesi ve anatomik pozisyonlama öznitelik vektörlerinin çıkarımı ve önbelleklenmesi
if os.path.exists(Q_CACHE_NPZ):
    q_data = np.load(Q_CACHE_NPZ)
    if (len(train_samples) > 0 and
        len(q_data['train_q']) == len(train_samples) and
        len(q_data['val_q']) == len(val_samples) and
        len(q_data['test_q']) == len(test_samples)):
        train_q = q_data['train_q']
        val_q   = q_data['val_q']
        test_q  = q_data['test_q']
        print(f'12-B Kalite/Pozisyonlama öznitelikleri önbellekten yüklendi: {Q_CACHE_NPZ}')
    else:
        os.remove(Q_CACHE_NPZ)

if not os.path.exists(Q_CACHE_NPZ):
    print('12-B Görüntü Kalitesi ve Pozisyonlama Öznitelikleri hesaplanıyor ve önbelleğe kaydediliyor...')
    def _compute_q_list(fpaths, desc):
        arr = np.zeros((len(fpaths), 12), dtype=np.float32)
        for i, fp in enumerate(tqdm(fpaths, desc=desc, leave=False)):
            im = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
            if im is not None:
                arr[i] = extract_quality_positioning_features(im)
        return arr
    train_q = _compute_q_list(train_samples, 'Train Q-Features')
    val_q   = _compute_q_list(val_samples,   'Val Q-Features')
    test_q  = _compute_q_list(test_samples,  'Test Q-Features')
    np.savez_compressed(Q_CACHE_NPZ, train_q=train_q, val_q=val_q, test_q=test_q)
    print(f'Kalite öznitelikleri kaydedildi: {Q_CACHE_NPZ}')

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.92, 1.08), translate_percent=(-0.05, 0.05), rotate=(-12, 12), border_mode=cv2.BORDER_CONSTANT, p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class MammosightCachedDataset(Dataset):
    def __init__(self, samples, labels, q_feats, transform=None):
        self.samples   = samples
        self.labels    = labels
        self.q_feats   = q_feats
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img = cv2.imread(self.samples[idx])
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(self.q_feats[idx], dtype=torch.float32), self.labels[idx]

train_ds = MammosightCachedDataset(train_samples, train_labels, train_q, transform=train_transform)
val_ds   = MammosightCachedDataset(val_samples,   val_labels,   val_q,   transform=val_transform)
test_ds  = MammosightCachedDataset(test_samples,  test_labels,  test_q,  transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)


In [ ]:
# ==============================================================================
# Bölüm 4: Genelleştirilmiş Ortalama Havuzlama (GeM) ve Kalite Öznitelik Füzyonu Mimarisi
# ==============================================================================
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(1.0 / self.p)

class MammosightEfficientNet(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_classes=3, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, num_classes=0, global_pool='', drop_path_rate=0.2
        )
        in_features = self.backbone.num_features
        self.gem = GeM(p=3.0)
        self.maxpool = nn.AdaptiveMaxPool2d(1)

        self.quality_mlp = nn.Sequential(
            nn.Linear(12, 32),
            nn.BatchNorm1d(32),
            nn.SiLU(inplace=True),
            nn.Linear(32, 32),
            nn.BatchNorm1d(32),
            nn.SiLU(inplace=True),
        )

        self.neck = nn.Sequential(
            nn.Linear(in_features * 2 + 32, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(inplace=True),
        )
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in [0.2, 0.3, 0.4, 0.5, 0.5]])
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x, q_feat):
        feat_map = self.backbone(x)
        f_gem = self.gem(feat_map).flatten(1)
        f_max = self.maxpool(feat_map).flatten(1)
        f_q   = self.quality_mlp(q_feat)
        emb   = self.neck(torch.cat([f_gem, f_max, f_q], dim=1))
        if self.training:
            logits = torch.stack([self.fc(drop(emb)) for drop in self.dropouts], dim=0).mean(dim=0)
        else:
            logits = self.fc(emb)
        return logits

class FocalCrossEntropyLoss(nn.Module):
    def __init__(self, weight=None, gamma=1.5, label_smoothing=0.05):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce_loss)
        focal_term = (1.0 - pt) ** self.gamma
        loss = focal_term * ce_loss
        if self.weight is not None:
            loss = loss * self.weight[targets]
        return loss.mean()

model = MammosightEfficientNet('efficientnet_b0', num_classes=3, pretrained=True).to(DEVICE)
model_ema = ModelEmaV2(model, decay=0.9995, device=DEVICE)

if NUM_GPUS > 1:
    model = nn.DataParallel(model)


In [ ]:
# ==============================================================================
# Bölüm 5: Model Eğitimi, ModelEMA ve Kontrol Noktası Yönetimi
# ==============================================================================
EPOCHS   = 20
PATIENCE = 6
LR       = 3e-4
WEIGHT_DECAY = 1e-2

class_weights = torch.tensor([1.05, 1.00, 1.15], dtype=torch.float32, device=DEVICE)
criterion = FocalCrossEntropyLoss(weight=class_weights, gamma=1.5, label_smoothing=0.05)

backbone_params, head_params = [], []
for name, param in model.named_parameters():
    if 'backbone' in name:
        backbone_params.append(param)
    else:
        head_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': LR * 0.5},
    {'params': head_params,     'lr': LR * 1.5},
], weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = GradScaler()

start_epoch   = 1
best_val_auc  = 0.0
best_val_acc  = 0.0
no_improve    = 0
history       = []
is_completed  = False

if os.path.exists(STATE_JSON):
    try:
        with open(STATE_JSON, 'r') as f:
            st = json.load(f)
            is_completed = bool(st.get('completed', False)) and os.path.exists(BEST_MODEL_PT)
    except Exception:
        is_completed = False

if os.path.exists(RESULTS_CSV):
    try:
        history = pd.read_csv(RESULTS_CSV).to_dict('records')
    except Exception:
        history = []

if is_completed:
    print(f'Eğitim önceki oturumda tamamen bitmiş ({BEST_MODEL_PT})! Eğitim döngüsü atlanıyor...')
else:
    if os.path.exists(LAST_CKPT):
        print(f'Önceki oturumdan kalan checkpoint bulundu ({LAST_CKPT}), kaldığı epoch\'tan devam ediliyor...')
        ckpt = torch.load(LAST_CKPT, map_location=DEVICE)
        raw_model = model.module if isinstance(model, nn.DataParallel) else model
        raw_model.load_state_dict(ckpt['model_state_dict'])
        model_ema.module.load_state_dict(ckpt['ema_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if 'scaler_state_dict' in ckpt:
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        start_epoch  = ckpt['epoch'] + 1
        best_val_auc = ckpt.get('best_val_auc', 0.0)
        best_val_acc = ckpt.get('best_val_acc', 0.0)
        no_improve   = ckpt.get('no_improve', 0)
        print(f'Epoch {start_epoch}/{EPOCHS} noktasından devam ediliyor (En İyi Val Macro AUC: {best_val_auc:.4f})')

    y_val_bin = label_binarize(val_labels, classes=[0, 1, 2])

    for epoch in range(start_epoch, EPOCHS + 1):
        if time.time() - SESSION_START_TIME > MAX_SESSION_SECONDS:
            print(f'Kaggle 11.2 saatlik güvenli oturum sınırına ulaşıldı! Epoch {epoch-1} sonunda güvenli çıkış yapılıyor.')
            break

        t0 = time.time()
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch:02d}/{EPOCHS}', leave=False)
        for imgs, q_feats, labels in pbar:
            imgs    = imgs.to(DEVICE, non_blocking=True)
            q_feats = q_feats.to(DEVICE, non_blocking=True)
            labels  = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with autocast():
                outputs = model(imgs, q_feats)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            scaler.step(optimizer)
            scaler.update()
            model_ema.update(model.module if isinstance(model, nn.DataParallel) else model)

            train_loss    += loss.item() * imgs.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            train_total   += imgs.size(0)
            pbar.set_postfix(loss=f'{train_loss/train_total:.4f}', acc=f'{train_correct/train_total*100:.1f}%')

        scheduler.step()
        train_loss /= train_total
        train_acc   = train_correct / train_total

        # Üstel Hareketli Ortalama (ModelEMA) ağırlıkları ile doğrulama değerlendirmesi
        eval_net = model_ema.module
        eval_net.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        val_probs_list = []
        with torch.no_grad():
            for imgs, q_feats, labels in val_loader:
                imgs    = imgs.to(DEVICE, non_blocking=True)
                q_feats = q_feats.to(DEVICE, non_blocking=True)
                labels  = labels.to(DEVICE, non_blocking=True)
                with autocast():
                    outputs = eval_net(imgs, q_feats)
                    loss = criterion(outputs, labels)
                    probs = F.softmax(outputs, dim=1)
                val_loss    += loss.item() * imgs.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total   += imgs.size(0)
                val_probs_list.append(probs.cpu().numpy())

        val_loss /= val_total
        val_acc   = val_correct / val_total
        val_probs_ep = np.vstack(val_probs_list)
        val_auc = roc_auc_score(y_val_bin, val_probs_ep, multi_class='ovr', average='macro')
        elapsed = time.time() - t0

        history.append({
            'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
            'train_acc': train_acc, 'val_acc': val_acc, 'val_auc': val_auc,
            'lr': optimizer.param_groups[0]['lr']
        })
        pd.DataFrame(history).to_csv(RESULTS_CSV, index=False)

        print(f'Epoch {epoch:02d}/{EPOCHS} ({elapsed:.0f}s) | '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}% Macro-AUC: {val_auc:.4f}')

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_val_acc = val_acc
            no_improve = 0
            torch.save(eval_net.state_dict(), BEST_MODEL_PT)
            print(f'Yeni en iyi model checkpointi kaydedildi: (Val Macro-AUC: {best_val_auc:.4f}, Val Acc: {best_val_acc*100:.2f}%)')
        else:
            no_improve += 1

        # Eğitim döngüsü boyunca periyodik kontrol noktası (checkpoint) durumu kaydı
        raw_model = model.module if isinstance(model, nn.DataParallel) else model
        torch.save({
            'epoch': epoch,
            'model_state_dict': raw_model.state_dict(),
            'ema_state_dict': eval_net.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_val_auc': best_val_auc,
            'best_val_acc': best_val_acc,
            'no_improve': no_improve,
        }, LAST_CKPT)

        if no_improve >= PATIENCE:
            print(f'Early stopping! {PATIENCE} epoch boyunca Val Macro-AUC iyileşmedi.')
            with open(STATE_JSON, 'w') as f:
                json.dump({'completed': True, 'epoch': epoch, 'best_val_auc': float(best_val_auc)}, f)
            break

        if epoch == EPOCHS:
            with open(STATE_JSON, 'w') as f:
                json.dump({'completed': True, 'epoch': epoch, 'best_val_auc': float(best_val_auc)}, f)


In [ ]:
# ==============================================================================
# Bölüm 6: Çok Ölçekli TTA, Hasta Düzeyinde Çoklu Projeksiyon Füzyonu ve Pareto Kalibrasyonu
# ==============================================================================
best_net = MammosightEfficientNet('efficientnet_b0', num_classes=3, pretrained=False).to(DEVICE)
best_net.load_state_dict(torch.load(BEST_MODEL_PT, map_location=DEVICE))
best_net.eval()

if os.path.exists(TTA_CACHE_NPZ) and os.path.exists(STATE_JSON):
    print(f'Kayıtlı TTA tahmin önbelleği yüklendi: {TTA_CACHE_NPZ}')
    cached = np.load(TTA_CACHE_NPZ)
    val_raw, val_probs_tta   = cached['val_raw'], cached['val_probs_tta']
    test_raw, test_probs_tta = cached['test_raw'], cached['test_probs_tta']
else:
    def predict_4view_tta(loader):
        p_raw_list, p_tta_list = [], []
        with torch.no_grad():
            for imgs, q_feats, _ in tqdm(loader, desc='4-Yönlü TTA', leave=False):
                imgs    = imgs.to(DEVICE, non_blocking=True)
                q_feats = q_feats.to(DEVICE, non_blocking=True)
                imgs_flip = torch.flip(imgs, dims=[3])
                imgs_576  = F.interpolate(imgs, size=(576, 576), mode='bilinear', align_corners=False)
                imgs_576_flip = torch.flip(imgs_576, dims=[3])

                with autocast():
                    p1 = F.softmax(best_net(imgs, q_feats), dim=1)
                    p2 = F.softmax(best_net(imgs_flip, q_feats), dim=1)
                    p3 = F.softmax(best_net(imgs_576, q_feats), dim=1)
                    p4 = F.softmax(best_net(imgs_576_flip, q_feats), dim=1)

                p_blend = 0.35 * p1 + 0.25 * p2 + 0.25 * p3 + 0.15 * p4
                p_raw_list.append(p1.cpu().numpy())
                p_tta_list.append(p_blend.cpu().numpy())

        return np.vstack(p_raw_list), np.vstack(p_tta_list)

    print('Validation ve Test kümeleri üzerinde 4-Yönlü Multi-Scale TTA hesaplanıyor ve önbelleğe kaydediliyor...')
    val_raw, val_probs_tta   = predict_4view_tta(val_loader)
    test_raw, test_probs_tta = predict_4view_tta(test_loader)
    np.savez_compressed(
        TTA_CACHE_NPZ,
        val_raw=val_raw, val_probs_tta=val_probs_tta,
        test_raw=test_raw, test_probs_tta=test_probs_tta
    )
    print(f'TTA önbelleği diske kaydedildi: {TTA_CACHE_NPZ}')

y_val  = val_labels
y_true = test_labels
y_val_bin = label_binarize(y_val, classes=[0, 1, 2])

# Hasta Düzeyinde Çoklu Projeksiyon Bağlamsal Füzyonu (Patient-Level Multi-View Contextual Fusion)
def apply_patient_multiview_context(probs, cases, views, gamma_mv=0.45):
    """Hasta düzeyinde çoklu projeksiyon (RCC, LCC, RMLO, LMLO) bağlamsal füzyon fonksiyonu.

    Klinik mamografi değerlendirme protokollerine uygun olarak tekil görüntü olasılıklarını,
    ipsilateral, kontralateral ve bilateral projeksiyon havuzlama çıktıları ile harmanlar.
    """
    if (cases >= 0).sum() < len(cases) * 0.5:
        return probs
    out_p = probs.copy()
    df_c = pd.DataFrame({'case': cases, 'view': views, 'p0': probs[:, 0], 'p1': probs[:, 1], 'p2': probs[:, 2]})
    case_mean = df_c.groupby('case')[['p0', 'p1', 'p2']].transform('mean').values
    case_max  = df_c.groupby('case')[['p0', 'p1', 'p2']].transform('max').values

    patient_context = np.zeros_like(probs)
    patient_context[:, 0] = 0.70 * case_mean[:, 0] + 0.30 * case_max[:, 0]
    patient_context[:, 1] = 0.75 * case_mean[:, 1] + 0.25 * case_max[:, 1]
    patient_context[:, 2] = 0.45 * case_mean[:, 2] + 0.55 * case_max[:, 2]
    patient_context /= np.clip(patient_context.sum(axis=1, keepdims=True), 1e-8, None)

    fused = (1.0 - gamma_mv) * out_p + gamma_mv * patient_context
    return fused / np.clip(fused.sum(axis=1, keepdims=True), 1e-8, None)

base_v_auc = roc_auc_score(y_val_bin, val_probs_tta, multi_class='ovr', average='macro')
best_gamma, best_mv_auc = 0.0, base_v_auc
for g_cand in np.linspace(0.0, 0.75, 16):
    cand_v = apply_patient_multiview_context(val_probs_tta, val_cases, val_views, gamma_mv=g_cand)
    c_auc = roc_auc_score(y_val_bin, cand_v, multi_class='ovr', average='macro')
    if c_auc > best_mv_auc:
        best_mv_auc = c_auc
        best_gamma = g_cand

print(f'Optimal Multi-View Bağlam Ağırlığı: gamma={best_gamma:.2f} (Val Macro AUC: {base_v_auc:.4f} -> {best_mv_auc:.4f})')
val_probs_mv  = apply_patient_multiview_context(val_probs_tta, val_cases, val_views, gamma_mv=best_gamma)
test_probs_mv = apply_patient_multiview_context(test_probs_tta, test_cases, test_views, gamma_mv=best_gamma)

def apply_calibration(probs, params):
    w0, w1, w2 = params
    logits = np.log(np.clip(probs, 1e-8, 1.0))
    logits[:, 0] += w0; logits[:, 1] += w1; logits[:, 2] += w2
    exp_l = np.exp(logits - logits.max(axis=1, keepdims=True))
    return exp_l / exp_l.sum(axis=1, keepdims=True)

def optimize_pareto_calibration(val_p, y_v):
    """Klinik karar eşiklerinin Pareto-optimal kalibrasyonu.

    Yüksek riskli BI-RADS 4-5 sınıfında yanlış negatifliği asgariye indirmek amacıyla
    duyarlılık (sensitivity) ile genel makro F1 skorunu eşzamanlı optimize eder.
    """
    raw_preds_v = val_p.argmax(axis=1)
    raw_rec_45  = ((raw_preds_v == 2) & (y_v == 2)).sum() / max((y_v == 2).sum(), 1)
    raw_f1_0    = f1_score(y_v, raw_preds_v, average=None)[0]

    def _obj(params):
        cal_v = apply_calibration(val_p, params)
        preds_v = cal_v.argmax(axis=1)
        f1_c = f1_score(y_v, preds_v, average=None)
        macro_f1 = f1_c.mean()
        acc = accuracy_score(y_v, preds_v)
        tp_45 = ((preds_v == 2) & (y_v == 2)).sum(); fn_45 = ((preds_v != 2) & (y_v == 2)).sum()
        tn_45 = ((preds_v != 2) & (y_v != 2)).sum(); fp_45 = ((preds_v == 2) & (y_v != 2)).sum()
        rec_45  = tp_45 / max(tp_45 + fn_45, 1)
        spec_45 = tn_45 / max(tn_45 + fp_45, 1)
        bal_45  = 0.5 * (rec_45 + spec_45)

        penalty = 0.0
        if rec_45 < raw_rec_45 + 0.015:
            penalty += 6.0 * (raw_rec_45 + 0.015 - rec_45)
        if f1_c[0] < raw_f1_0:
            penalty += 5.0 * (raw_f1_0 - f1_c[0])
        reg = 0.015 * (params[0]**2 + params[1]**2 + params[2]**2)
        return -(0.40 * macro_f1 + 0.30 * acc + 0.30 * bal_45) + penalty + reg

    best_p = [0.0, 0.0, 0.0]
    best_val = _obj(best_p)
    for iw0 in [-0.08, -0.04, 0.0, 0.04, 0.08]:
        for iw2 in [0.0, 0.05, 0.10, 0.15, 0.20]:
            res = minimize(_obj, x0=[iw0, 0.0, iw2], method='Nelder-Mead', options={'maxiter': 500})
            if res.fun < best_val:
                best_val = res.fun
                best_p = res.x
    return best_p

best_params = optimize_pareto_calibration(val_probs_mv, y_val)
y_scores = apply_calibration(test_probs_mv, best_params)
y_pred   = y_scores.argmax(axis=1)
acc      = (y_true == y_pred).mean() * 100

print(f'Klinik Kalibrasyon Ağırlıkları: {[round(x, 4) for x in best_params]}')
print('\n Sınıflandırma Raporu (GeM + Quality-MLP + Multi-View + Calibration):')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))
print(f'Test Doğruluğu: {acc:.2f}%')


In [ ]:
# ==============================================================================
# Bölüm 7: Ablasyon ve Karşılaştırma Analizi (Tablo 4: EfficientNet-B0)
# ==============================================================================
def compute_paper_table4_metrics(y_t, probs, preds=None):
    """Makale Tablo 4 için tanısal ve istatistiksel performans metriklerini hesaplar.

    Metrikler: Doğruluk, Makro/Sınıf Bazlı Hassasiyet, Duyarlılık, F1-Skoru, Makro ROC-AUC
    ve klinik sevk odaklı BI-RADS 4-5 duyarlılık oranı.
    """
    if preds is None:
        preds = probs.argmax(axis=1)
    y_bin = label_binarize(y_t, classes=[0, 1, 2])
    auc_0  = roc_auc_score(y_bin[:, 0], probs[:, 0])
    auc_12 = roc_auc_score(y_bin[:, 1], probs[:, 1])
    auc_45 = roc_auc_score(y_bin[:, 2], probs[:, 2])
    auc_macro = roc_auc_score(y_bin, probs, multi_class='ovr', average='macro')

    diag_mask = (y_t != 0) & (preds != 0)
    diag_acc = (y_t[diag_mask] == preds[diag_mask]).mean() * 100.0 if diag_mask.sum() > 0 else 0.0

    tp_45 = ((preds == 2) & (y_t == 2)).sum(); fn_45 = ((preds != 2) & (y_t == 2)).sum()
    tn_45 = ((preds != 2) & (y_t != 2)).sum(); fp_45 = ((preds == 2) & (y_t != 2)).sum()
    rec_45  = (tp_45 / max(tp_45 + fn_45, 1)) * 100.0
    spec_45 = (tn_45 / max(tn_45 + fp_45, 1)) * 100.0
    prec_45 = (tp_45 / max(tp_45 + fp_45, 1)) * 100.0
    f1_c = f1_score(y_t, preds, average=None)

    return {
        'BI-RADS 4-5 ROC-AUC': round(auc_45, 4),
        'BI-RADS 1-2 ROC-AUC': round(auc_12, 4),
        'BI-RADS 0 ROC-AUC': round(auc_0, 4),
        'ROC-AUC (Macro)': round(auc_macro, 4),
        '2x2 Tanısal Alt-Matris Doğruluğu (1-2 vs 4-5) (%)': round(diag_acc, 2),
        'BI-RADS 4-5 Dengeli Doğruluk (%)': round(0.5 * (rec_45 + spec_45), 2),
        'BI-RADS 4-5 Özgüllük / Specificity (%)': round(spec_45, 2),
        'BI-RADS 4-5 Kesinlik / Precision (%)': round(prec_45, 2),
        'BI-RADS 4-5 Duyarlılık / Recall (%)': round(rec_45, 2),
        'BI-RADS 4-5 F1-Skoru': round(f1_c[2], 4),
        'BI-RADS 1-2 F1-Skoru': round(f1_c[1], 4),
        'BI-RADS 0 F1-Skoru': round(f1_c[0], 4),
        '3-Sınıflı Genel Doğruluk (Test Accuracy) (%)': round((y_t == preds).mean() * 100.0, 2),
        'Makro F1-Skoru (3-Sınıf)': round(f1_score(y_t, preds, average='macro'), 4),
    }

paper_baseline = {
    'BI-RADS 4-5 ROC-AUC': 0.7790,
    'BI-RADS 1-2 ROC-AUC': 0.7510,
    'BI-RADS 0 ROC-AUC': 0.6920,
    'ROC-AUC (Macro)': 0.7400,
    '2x2 Tanısal Alt-Matris Doğruluğu (1-2 vs 4-5) (%)': 74.80,
    'BI-RADS 4-5 Dengeli Doğruluk (%)': 70.11,
    'BI-RADS 4-5 Özgüllük / Specificity (%)': 85.46,
    'BI-RADS 4-5 Kesinlik / Precision (%)': 70.07,
    'BI-RADS 4-5 Duyarlılık / Recall (%)': 54.76,
    'BI-RADS 4-5 F1-Skoru': 0.6100,
    'BI-RADS 1-2 F1-Skoru': 0.6300,
    'BI-RADS 0 F1-Skoru': 0.4100,
    '3-Sınıflı Genel Doğruluk (Test Accuracy) (%)': 57.09,
    'Makro F1-Skoru (3-Sınıf)': 0.5500,
}

comp_df = pd.DataFrame({
    'Makale Baz (EfficientNet-B0)': paper_baseline,
    'Yeni: GeM+Q-MLP (Tekil)': compute_paper_table4_metrics(y_true, test_raw),
    'Yeni: 4-Yön TTA': compute_paper_table4_metrics(y_true, test_probs_tta),
    'Yeni: Multi-View + Kalibre': compute_paper_table4_metrics(y_true, y_scores, y_pred),
})
comp_df['Kazanç (Yeni - Baz)'] = (comp_df['Yeni: Multi-View + Kalibre'] - comp_df['Makale Baz (EfficientNet-B0)']).round(4)

print('\n' + '=' * 110)
print('TABLO 4 — KARŞILAŞTIRMALI PERFORMANS ANALİZİ (EfficientNet-B0)')
print('=' * 110)
print(comp_df.to_string())
print('=' * 110)
comp_df.to_csv(os.path.join(SAVE_DIR, 'efficientnet_table4_comparison.csv'))


In [ ]:
# ==============================================================================
# Bölüm 8: Tanısal Metrikler, Hata Matrisi ve ROC Eğrisi Görselleştirmeleri
# ==============================================================================
COLORS = ['#6C63FF', '#FF6584', '#43C59E']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#1a1a2e')

ax = axes[0]
ax.set_facecolor('#16213e')
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(DISPLAY_NAMES, fontsize=11, color='white', rotation=15)
ax.set_yticklabels(DISPLAY_NAMES, fontsize=11, color='white')
ax.set_xlabel('Tahmin Edilen', fontsize=12, color='white')
ax.set_ylabel('Gerçek', fontsize=12, color='white')
ax.set_title('Confusion Matrix (Normalize)', fontsize=14, color='white', pad=15)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#444')
for i in range(3):
    for j in range(3):
        color = 'white' if cm_norm[i, j] < 0.5 else '#1a1a2e'
        ax.text(j, i, f'{cm_norm[i,j]:.2f}\n({cm[i,j]})',
                ha='center', va='center', fontsize=10, color=color, fontweight='bold')

ax2 = axes[1]
ax2.set_facecolor('#16213e')
y_bin = label_binarize(y_true, classes=[0, 1, 2])
for i, (cls_name, color) in enumerate(zip(DISPLAY_NAMES, COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_scores[:, i])
    roc_auc = auc(fpr, tpr)
    ax2.plot(fpr, tpr, color=color, lw=2.5, label=f'{cls_name} (AUC = {roc_auc:.3f})')
ax2.plot([0, 1], [0, 1], 'w--', lw=1, alpha=0.5)
ax2.set_xlim([0.0, 1.0]); ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('False Positive Rate', fontsize=12, color='white')
ax2.set_ylabel('True Positive Rate', fontsize=12, color='white')
ax2.set_title('ROC Eğrileri (One-vs-Rest)', fontsize=14, color='white', pad=15)
ax2.tick_params(colors='white')
for spine in ax2.spines.values(): spine.set_edgecolor('#444')
ax2.legend(loc='lower right', fontsize=11, facecolor='#1a1a2e', labelcolor='white', edgecolor='#555')

plt.suptitle(f'EfficientNet-B0 (GeM+Q-MLP+Multi-View) — MammosighTR BI-RADS | Test Doğruluğu: {acc:.2f}%',
             fontsize=15, color='white', y=1.02, fontweight='bold')
plt.tight_layout()
out_fig = '/kaggle/working/efficientnet_results.png' if os.path.exists('/kaggle') else './efficientnet_results.png'
plt.savefig(out_fig, dpi=180, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

if os.path.exists(RESULTS_CSV):
    df_res = pd.read_csv(RESULTS_CSV)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#1a1a2e')
    ax = axes[0]; ax.set_facecolor('#16213e')
    ax.plot(df_res['epoch'], df_res['train_loss'], label='Train Loss', color='#6C63FF', lw=2)
    ax.plot(df_res['epoch'], df_res['val_loss'],   label='Val Loss',   color='#FF6584', lw=2)
    ax.set_xlabel('Epoch', color='white'); ax.set_ylabel('Loss', color='white')
    ax.set_title('Eğitim / Doğrulama Loss', color='white', fontsize=13)
    ax.tick_params(colors='white'); ax.legend(facecolor='#1a1a2e', labelcolor='white', edgecolor='#555')
    for spine in ax.spines.values(): spine.set_edgecolor('#444')

    ax2 = axes[1]; ax2.set_facecolor('#16213e')
    ax2.plot(df_res['epoch'], df_res['train_acc'] * 100, label='Train Acc (%)', color='#6C63FF', lw=2)
    ax2.plot(df_res['epoch'], df_res['val_acc'] * 100,   label='Val Acc (%)',   color='#43C59E', lw=2)
    ax2.plot(df_res['epoch'], df_res['val_auc'] * 100,   label='Val Macro-AUC (x100)', color='#FFC75F', lw=2, linestyle='--')
    ax2.set_xlabel('Epoch', color='white'); ax2.set_ylabel('Skor (%)', color='white')
    ax2.set_title('Accuracy & Macro-AUC Eğrileri', color='white', fontsize=13)
    ax2.tick_params(colors='white'); ax2.legend(facecolor='#1a1a2e', labelcolor='white', edgecolor='#555')
    for spine in ax2.spines.values(): spine.set_edgecolor('#444')

    plt.suptitle('EfficientNet-B0 — Eğitim Eğrileri', color='white', fontsize=14, fontweight='bold')
    plt.tight_layout()
    curve_fig = '/kaggle/working/efficientnet_training_curves.png' if os.path.exists('/kaggle') else './efficientnet_training_curves.png'
    plt.savefig(curve_fig, dpi=180, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
